In [ ]:
#img_url, product_name, category, sub_category, brand, rating, review_count, discount_rate, price, style_tag, url

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from deep_translator import GoogleTranslator
import pandas as pd, time, re

# =========================
# ⚙️ 설정값
# =========================
RANKING_URL = "https://www.musinsa.com/main/musinsa/ranking?gf=A&storeCode=musinsa&sectionId=199&contentsId=&categoryCode=003000&ageBand=AGE_BAND_ALL&subPan=product&period=MONTHLY"
TARGET_URLS =  1000
WAIT_TIME = 8
STEP_SLEEP = 1.1
NO_GAIN_LIMIT = 6

translator = GoogleTranslator(source='ko', target='en')

def translate_auto(text):
    """한국어를 영어로 번역 (deep_translator 사용)"""
    if not text:
        return None
    try:
        return translator.translate(text)
    except:
        return text

def clean_number(text):
    if not text:
        return None
    digits = re.sub(r"[^0-9]", "", text)
    return int(digits) if digits else None

# =========================
# 1️⃣ 드라이버 세팅
# =========================
options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--disable-dev-shm-usage")
# options.add_argument("--headless")  # 필요 시 활성화
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, WAIT_TIME)

# =========================
# 2️⃣ 상품 URL 수집 (점진적 스크롤)
# =========================
driver.get(RANKING_URL)
time.sleep(5)

collected = set()
no_gain_count = 0
last_count = 0

def current_links():
    hrefs = driver.execute_script("""
        return Array.from(document.querySelectorAll('a[href*="/products/"]'))
                     .map(a => a.href.split('?')[0]);
    """)
    return [h for h in hrefs if "/products/" in h]

print("🚀 상품 URL 수집 시작...")

for step in range(200):
    driver.execute_script("window.scrollBy(0, window.innerHeight * 0.9);")
    time.sleep(STEP_SLEEP)
    for link in current_links():
        collected.add(link)

    current_count = len(collected)
    gain = current_count - last_count
    print(f"📦 Step {step+1:03d} ▶ 누적 {current_count}개 (+{gain})")

    if gain == 0:
        no_gain_count += 1
    else:
        no_gain_count = 0
    last_count = current_count

    if no_gain_count >= NO_GAIN_LIMIT or current_count >= TARGET_URLS:
        print("✅ 충분히 로드됨 — 스크롤 중단")
        break

product_urls = list(collected)[:TARGET_URLS]
print(f"\n🎯 최종 URL 수집: {len(product_urls)}개")

# =========================
# 3️⃣ 안전 추출 함수
# =========================
def safe_get_text(selector):
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
    except:
        return None

def safe_get_attr(selector, attr):
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).get_attribute(attr)
    except:
        return None

# =========================
# 4️⃣ 상세 페이지 크롤링
# =========================
data = []
for idx, url in enumerate(product_urls, 1):
    driver.get(url)
    time.sleep(1.8)

    row = {
        "img_url": safe_get_attr("#root div.Swiper__Wrap-sc-uxvjgl-0 img", "src"),
        "product_name": safe_get_text("#root div.GoodsName__Wrap-sc-1omefes-0 span"),
        "category": safe_get_text("#root div.Category__Wrap-sc-1prswe3-1 span:nth-child(1) a"),
        "sub_category": safe_get_text("#root div.Category__Wrap-sc-1prswe3-1 span:nth-child(2) a"),
        "brand": safe_get_text("#root div.Brand__Wrap-sc-12cqkwk-0 a span span"),
        "rating": safe_get_text("#root div.Review__Wrap-sc-hw7d9p-0 span.text-body_13px_med"),
        "review_count": clean_number(safe_get_text("#root div.Review__Wrap-sc-hw7d9p-0 span.text-body_13px_reg.underline")),
        "discount_rate": clean_number(safe_get_text("#root div.Price__CurrentPrice-sc-1hw5bl8-6 span.text-red")),
        "price": clean_number(safe_get_text("#root div.Price__CurrentPrice-sc-1hw5bl8-6 span.text-black")),
        "style_tag": safe_get_text("#root div.ProductTags__Wrap-sc-1eb70kd-0 ul"),
        "url": url
    }
    data.append(row)

    if idx % 50 == 0:
        df_temp = pd.DataFrame(data)
        for col in ["category", "sub_category", "brand"]:
            df_temp[col] = df_temp[col].apply(translate_auto)
        df_temp.to_excel(f"musinsa_Bottom_{idx}.xlsx", index=False)
        print(f"💾 {idx}개 중간저장 완료")

# =========================
# 5️⃣ 저장 (영문 변환 포함)
# =========================
driver.quit()
df = pd.DataFrame(data)

for col in ["category", "sub_category", "brand"]:
    df[col] = df[col].apply(translate_auto)

df.to_excel("C01_1000_Bottom.xlsx", index=False)
print("\n✅ 저장 완료 → C01_1000_Bottom.xlsx (영문 변환 완료)")

🚀 상품 URL 수집 시작...
📦 Step 001 ▶ 누적 23개 (+23)
📦 Step 002 ▶ 누적 41개 (+18)
📦 Step 003 ▶ 누적 41개 (+0)
📦 Step 004 ▶ 누적 59개 (+18)
📦 Step 005 ▶ 누적 77개 (+18)
📦 Step 006 ▶ 누적 77개 (+0)
📦 Step 007 ▶ 누적 95개 (+18)
📦 Step 008 ▶ 누적 95개 (+0)
📦 Step 009 ▶ 누적 113개 (+18)
📦 Step 010 ▶ 누적 113개 (+0)
📦 Step 011 ▶ 누적 131개 (+18)
📦 Step 012 ▶ 누적 131개 (+0)
📦 Step 013 ▶ 누적 149개 (+18)
📦 Step 014 ▶ 누적 149개 (+0)
📦 Step 015 ▶ 누적 167개 (+18)
📦 Step 016 ▶ 누적 185개 (+18)
📦 Step 017 ▶ 누적 185개 (+0)
📦 Step 018 ▶ 누적 203개 (+18)
📦 Step 019 ▶ 누적 203개 (+0)
📦 Step 020 ▶ 누적 221개 (+18)
📦 Step 021 ▶ 누적 221개 (+0)
📦 Step 022 ▶ 누적 239개 (+18)
📦 Step 023 ▶ 누적 239개 (+0)
📦 Step 024 ▶ 누적 257개 (+18)
📦 Step 025 ▶ 누적 257개 (+0)
📦 Step 026 ▶ 누적 275개 (+18)
📦 Step 027 ▶ 누적 293개 (+18)
📦 Step 028 ▶ 누적 293개 (+0)
📦 Step 029 ▶ 누적 311개 (+18)
📦 Step 030 ▶ 누적 311개 (+0)
📦 Step 031 ▶ 누적 329개 (+18)
📦 Step 032 ▶ 누적 329개 (+0)
📦 Step 033 ▶ 누적 347개 (+18)
📦 Step 034 ▶ 누적 347개 (+0)
📦 Step 035 ▶ 누적 365개 (+18)
📦 Step 036 ▶ 누적 365개 (+0)
📦 Step 037 ▶ 누적 383개 (+18)
📦 Step 